In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
import os
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [2]:
df = pd.read_parquet("cicids2017_cleaned.parquet")

print(f"Raw dataset shape: {df.shape}")
print(df.columns.str.strip())
print(f"Label distribution: {df['Label'].value_counts()}")

Raw dataset shape: (2313810, 78)
Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2313810 entries, 0 to 2313809
Data columns (total 78 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Protocol                  int8   
 1   Flow Duration             int32  
 2   Total Fwd Packets         int32  
 3   Total Backward Packets    int32  
 4   Fwd Packets Length Total  int32  
 5   Bwd Packets Length Total  int32  
 6   Fwd Packet Length Max     int16  
 7   Fwd Packet Length Min     int16  
 8   Fwd Packet Length Mean    float32
 9   Fwd Packet Length Std     float32
 10  Bwd Packet Length Max     int16  
 11  Bwd Packet Length Min     int16  
 12  Bwd Packet Length Mean    float32
 13  Bwd Packet Length Std     float32
 14  Flow Bytes/s              float64
 15  Flow Packets/s            float64
 16  Flow IAT Mean             float32
 17  Flow IAT Std              float32
 18  Flow IAT Max              int32  
 19  Flow IAT Min              int32  
 20  Fwd IAT Total           

In [4]:
df.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,4,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,3,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,609,7,4,484,414,233,0,69.14286,111.967896,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [5]:
cols_to_drop = [c for c in df.columns if "timestamp" in c.lower() or "time" in c.lower()]
df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
print(f"Dropped time columns: {cols_to_drop}")

Dropped time columns: []


In [6]:
# Convert feature columns to numeric (Label kept as string)
feature_cols = [c for c in df.columns if c != "Label"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

In [7]:
feature_cols = [c for c in df.columns if c != 'Label']
zero_cols_check = [c for c in feature_cols if (df[c] == 0).all()]
print(f"Columns with all zero values: {zero_cols_check}")

# Also check near-zero (99%+ zeros)
near_zero = [c for c in feature_cols if (df[c] == 0).mean() > 0.99]
print(f"\nColumns with 99%+ zero values: {near_zero}")

# Show exact zero percentage for suspicious columns
suspicious = ['Bwd PSH Flags', 'Bwd URG Flags', 
              'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
              'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
print(f"\nZero % for dropped columns (before they were dropped):")
for col in suspicious:
    if col in df.columns:
        pct = (df[col] == 0).mean() * 100
        print(f"  {col}: {pct:.2f}% zeros")
    else:
        print(f"  {col}: already dropped")

Columns with all zero values: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Columns with 99%+ zero values: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Zero % for dropped columns (before they were dropped):
  Bwd PSH Flags: 100.00% zeros
  Bwd URG Flags: 100.00% zeros
  Fwd Avg Bytes/Bulk: 100.00% zeros
  Fwd Avg Packets/Bulk: 100.00% zeros
  Fwd Avg Bulk Rate: 100.00% zeros
  Bwd Avg Bytes/Bulk: 100.00% zeros
  Bwd Avg Packets/Bulk: 100.00% zeros
  Bwd Avg Bulk Rate: 100.00% zeros


In [8]:
# Drop zero-only columns
zero_cols = [c for c in feature_cols if (df[c] == 0).all()]
df.drop(columns=zero_cols, inplace=True)
print(f"Dropped {len(zero_cols)} zero-only columns: {zero_cols}")

Dropped 8 zero-only columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [9]:
# ── Fix -1 sentinel in window columns FIRST ──────────────────────
win_cols = ['Init Fwd Win Bytes', 'Init Bwd Win Bytes']
for col in win_cols:
    if col in df.columns:
        df[col] = df[col].replace(-1, 0)
        print(f"Fixed -1 values in '{col}'")

# Drop rows with inf/-inf
inf_mask = df.isin([np.inf, -np.inf]).any(axis=1)
df = df[~inf_mask].copy()
print(f"Dropped {inf_mask.sum()} rows with inf/-inf values")

# Drop rows with negative values
feature_cols = [c for c in df.columns if c != "Label"]
neg_mask = (df[feature_cols] < 0).any(axis=1)
df = df[~neg_mask].copy()
print(f"Dropped {neg_mask.sum()} rows with negative values")


Fixed -1 values in 'Init Fwd Win Bytes'
Fixed -1 values in 'Init Bwd Win Bytes'
Dropped 0 rows with inf/-inf values
Dropped 2843 rows with negative values


In [10]:
df.reset_index(drop=True, inplace=True)
print(f"Cleaned dataset shape: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts()}")

Cleaned dataset shape: (2310967, 70)
Label distribution:
Label
Benign              1974668
DoS Hulk             172688
DDoS                 127995
DoS GoldenEye         10281
FTP-Patator            5927
DoS slowloris          5385
DoS Slowhttptest       5228
SSH-Patator            3217
PortScan               1956
Web_Brute_Force        1470
Bot                    1437
XSS                     652
Infiltration             35
SQL_Injection            21
Heartbleed                7
Name: count, dtype: int64


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2310967 entries, 0 to 2310966
Data columns (total 70 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Protocol                  int8   
 1   Flow Duration             int32  
 2   Total Fwd Packets         int32  
 3   Total Backward Packets    int32  
 4   Fwd Packets Length Total  int32  
 5   Bwd Packets Length Total  int32  
 6   Fwd Packet Length Max     int16  
 7   Fwd Packet Length Min     int16  
 8   Fwd Packet Length Mean    float32
 9   Fwd Packet Length Std     float32
 10  Bwd Packet Length Max     int16  
 11  Bwd Packet Length Min     int16  
 12  Bwd Packet Length Mean    float32
 13  Bwd Packet Length Std     float32
 14  Flow Bytes/s              float64
 15  Flow Packets/s            float64
 16  Flow IAT Mean             float32
 17  Flow IAT Std              float32
 18  Flow IAT Max              int32  
 19  Flow IAT Min              int32  
 20  Fwd IAT Total           

In [11]:
df = df[df["Label"] != "Heartbleed"]

In [14]:
print(df["Label"].value_counts())

Label
Benign              1974668
DoS Hulk             172688
DDoS                 127995
DoS GoldenEye         10281
FTP-Patator            5927
DoS slowloris          5385
DoS Slowhttptest       5228
SSH-Patator            3217
PortScan               1956
Web_Brute_Force        1470
Bot                    1437
XSS                     652
Infiltration             35
SQL_Injection            21
Name: count, dtype: int64


In [19]:
flag_cols = [
    'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',
    'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count',
    'Fwd PSH Flags', 'Fwd URG Flags',
]

print("Flag column ranges:")
print(f"{'Column':<25} {'Min':>6} {'Max':>6} {'Unique values'}")
print("-" * 60)
for col in flag_cols:
    if col in df.columns:
        print(f"{col:<25} {df[col].min():>6} {df[col].max():>6}  {sorted(df[col].unique())}")


Flag column ranges:
Column                       Min    Max Unique values
------------------------------------------------------------
FIN Flag Count                 0      1  [0, 1]
SYN Flag Count                 0      1  [0, 1]
RST Flag Count                 0      1  [0, 1]
PSH Flag Count                 0      1  [0, 1]
ACK Flag Count                 0      1  [0, 1]
URG Flag Count                 0      1  [0, 1]
CWE Flag Count                 0      1  [0, 1]
ECE Flag Count                 0      1  [0, 1]
Fwd PSH Flags                  0      1  [0, 1]
Fwd URG Flags                  0      1  [0, 1]


In [20]:
skip_cols    = flag_cols + ['Label', 'Protocol']
feature_cols = [c for c in df.columns if c not in skip_cols]

In [21]:
print(df['Label'].isna().sum())

0


In [22]:
print(df['Label'].apply(type).value_counts())

Label
<class 'str'>    2310960
Name: count, dtype: int64


In [23]:
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print(f"Normalized {len(feature_cols)} continuous features.")
print(f"Skipped    {len(skip_cols)} flag/categorical columns.")
print(f"Final shape: {df.shape}")

Normalized 58 continuous features.
Skipped    12 flag/categorical columns.
Final shape: (2310960, 70)


In [24]:
X = df.drop(columns=['Label'])
y = df['Label']

print(f"Dataset shape before split: {X.shape}")
print(f"Class distribution before SMOTE:\n{y.value_counts()}\n")

Dataset shape before split: (2310960, 69)
Class distribution before SMOTE:
Label
Benign              1974668
DoS Hulk             172688
DDoS                 127995
DoS GoldenEye         10281
FTP-Patator            5927
DoS slowloris          5385
DoS Slowhttptest       5228
SSH-Patator            3217
PortScan               1956
Web_Brute_Force        1470
Bot                    1437
XSS                     652
Infiltration             35
SQL_Injection            21
Name: count, dtype: int64



In [25]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train size : {X_train.shape}")
print(f"Val size   : {X_val.shape}")
print(f"Test size  : {X_test.shape}")

Train size : (1617672, 69)
Val size   : (346644, 69)
Test size  : (346644, 69)


In [27]:
from imblearn.over_sampling import RandomOverSampler, BorderlineSMOTE
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np
import torch

# ── Step 1: RandomOverSampler for extremely tiny classes ─────────
ros_strategy = {
    'SQL_Injection': 30,
    'Infiltration' : 50,
}
ros = RandomOverSampler(random_state=42, sampling_strategy=ros_strategy)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
print("After RandomOverSampler:", Counter(y_ros))

# ── Step 2: BorderlineSMOTE to minimum 10,000 per class ──────────
sampling_strategy = {
    'DDoS'            : 89596,
    'DoS GoldenEye'   : 30000,
    'FTP-Patator'     : 30000,
    'DoS slowloris'   : 30000,
    'DoS Slowhttptest': 30000,
    'SSH-Patator'     : 15000,
    'PortScan'        : 15000,
    'Web_Brute_Force' : 10000,
    'Bot'             : 10000,
    'XSS'             : 10000,
    'Infiltration'    : 10000,
    'SQL_Injection'   : 10000
}
borderline = BorderlineSMOTE(
    random_state=42,
    k_neighbors=4,
    sampling_strategy=sampling_strategy
)
X_train_resampled, y_train_resampled = borderline.fit_resample(X_ros, y_ros)

# ── Step 3: Class weights for remaining imbalance ────────────────
classes = np.unique(y_train_resampled)
weights = compute_class_weight('balanced', classes=classes, y=y_train_resampled)
class_weights = torch.tensor(weights, dtype=torch.float32)

print(f"\nAfter BorderlineSMOTE: {X_train_resampled.shape}")
print(f"\nFinal distribution:")
total = len(y_train_resampled)
for label, count in sorted(Counter(y_train_resampled).items(),
                            key=lambda x: x[1], reverse=True):
    print(f"  {label:<20} {count:>10,}  ({count/total*100:.1f}%)")

After RandomOverSampler: Counter({'Benign': 1382268, 'DoS Hulk': 120882, 'DDoS': 89596, 'DoS GoldenEye': 7197, 'FTP-Patator': 4149, 'DoS slowloris': 3769, 'DoS Slowhttptest': 3660, 'SSH-Patator': 2252, 'PortScan': 1369, 'Web_Brute_Force': 1029, 'Bot': 1006, 'XSS': 456, 'Infiltration': 50, 'SQL_Injection': 30})

After BorderlineSMOTE: (1792746, 69)

Final distribution:
  Benign                1,382,268  (77.1%)
  DoS Hulk                120,882  (6.7%)
  DDoS                     89,596  (5.0%)
  FTP-Patator              30,000  (1.7%)
  DoS slowloris            30,000  (1.7%)
  DoS GoldenEye            30,000  (1.7%)
  DoS Slowhttptest         30,000  (1.7%)
  SSH-Patator              15,000  (0.8%)
  PortScan                 15,000  (0.8%)
  Web_Brute_Force          10,000  (0.6%)
  Bot                      10,000  (0.6%)
  XSS                      10,000  (0.6%)
  Infiltration             10,000  (0.6%)
  SQL_Injection            10,000  (0.6%)


In [28]:
train_df = pd.DataFrame(X_train_resampled, columns=X.columns)
train_df['Label'] = y_train_resampled

val_df = pd.DataFrame(X_val.values, columns=X.columns)
val_df['Label'] = y_val.values

test_df = pd.DataFrame(X_test.values, columns=X.columns)
test_df['Label'] = y_test.values

In [30]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1792746 entries, 0 to 1792745
Data columns (total 70 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Protocol                  int8   
 1   Flow Duration             float64
 2   Total Fwd Packets         float64
 3   Total Backward Packets    float64
 4   Fwd Packets Length Total  float64
 5   Bwd Packets Length Total  float64
 6   Fwd Packet Length Max     float64
 7   Fwd Packet Length Min     float64
 8   Fwd Packet Length Mean    float64
 9   Fwd Packet Length Std     float64
 10  Bwd Packet Length Max     float64
 11  Bwd Packet Length Min     float64
 12  Bwd Packet Length Mean    float64
 13  Bwd Packet Length Std     float64
 14  Flow Bytes/s              float64
 15  Flow Packets/s            float64
 16  Flow IAT Mean             float64
 17  Flow IAT Std              float64
 18  Flow IAT Max              float64
 19  Flow IAT Min              float64
 20  Fwd IAT Total           

In [31]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346644 entries, 0 to 346643
Data columns (total 70 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Protocol                  346644 non-null  float64
 1   Flow Duration             346644 non-null  float64
 2   Total Fwd Packets         346644 non-null  float64
 3   Total Backward Packets    346644 non-null  float64
 4   Fwd Packets Length Total  346644 non-null  float64
 5   Bwd Packets Length Total  346644 non-null  float64
 6   Fwd Packet Length Max     346644 non-null  float64
 7   Fwd Packet Length Min     346644 non-null  float64
 8   Fwd Packet Length Mean    346644 non-null  float64
 9   Fwd Packet Length Std     346644 non-null  float64
 10  Bwd Packet Length Max     346644 non-null  float64
 11  Bwd Packet Length Min     346644 non-null  float64
 12  Bwd Packet Length Mean    346644 non-null  float64
 13  Bwd Packet Length Std     346644 non-null  f

In [32]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346644 entries, 0 to 346643
Data columns (total 70 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Protocol                  346644 non-null  float64
 1   Flow Duration             346644 non-null  float64
 2   Total Fwd Packets         346644 non-null  float64
 3   Total Backward Packets    346644 non-null  float64
 4   Fwd Packets Length Total  346644 non-null  float64
 5   Bwd Packets Length Total  346644 non-null  float64
 6   Fwd Packet Length Max     346644 non-null  float64
 7   Fwd Packet Length Min     346644 non-null  float64
 8   Fwd Packet Length Mean    346644 non-null  float64
 9   Fwd Packet Length Std     346644 non-null  float64
 10  Bwd Packet Length Max     346644 non-null  float64
 11  Bwd Packet Length Min     346644 non-null  float64
 12  Bwd Packet Length Mean    346644 non-null  float64
 13  Bwd Packet Length Std     346644 non-null  f

In [29]:
os.makedirs("processed", exist_ok=True)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)

torch.save(class_weights, "class_weights.pt")

print("Saved:")
print(f"  train.csv  →  {train_df.shape}")
print(f"  val.csv    →  {val_df.shape}")
print(f"  test.csv   →  {test_df.shape}")

Saved:
  train.csv  →  (1792746, 70)
  val.csv    →  (346644, 70)
  test.csv   →  (346644, 70)
